# NB3: Full Turn Integration (NB1 + NB2)

**Purpose:** Wire the causal engine to the GM pipeline for a complete turn cycle.

**Tests:**
1. Full turn sequence: decay → multi-turn → lagged propagation → GM adjudication → RNG → transitions → propagation → logging
2. 5 sequential turns with alternating US/Iran actions produce coherent state trajectories
3. Feedback loop visible: US sanctions → perceived coercion rises → Iran proxy escalation → perceived threat rises
4. Mechanical deltas passed to GM (ADR-002 additive layers)
5. Total LLM cost for 5 turns < $1

**Pass criteria:** State trajectories are plausible when plotted. No variable blows up or collapses.

In [ ]:
import sys
sys.path.insert(0, '..')

import uuid
import json
from wargame.scenario import load_scenario, init_db
from wargame.engine import (
    get_current_turn, get_variable, get_all_variables,
    run_mechanical_phases, apply_action_transitions,
    resolve_action, generate_action_id, record_state_history,
    get_state_history, start_multi_turn_action,
)
from wargame.models import ActionIntent, AdjudicationPacket
from wargame.gm import (
    select_relevant_domain_models,
    compute_mechanical_base_rate,
    build_gm_messages,
    validate_adjudication,
    normalize_probabilities,
)
from llm_client import call_llm_structured

SCENARIO_PATH = '../scenarios/us_iran_2026.yaml'
GM_MODEL = 'gemini/gemini-2.5-flash'
TRACE_ID = f'nb3_full_turn_{uuid.uuid4().hex[:8]}'

spec = load_scenario(SCENARIO_PATH)
conn = init_db(spec)

valid_var_ids = {sv.id for sv in spec.state_variables}
valid_actor_ids = {a.id for a in spec.actors}

print(f'Scenario: {spec.meta.name}')
print(f'Trace ID: {TRACE_ID}')

## Define the Turn Executor

A single function that runs a complete turn: mechanical phases → GM adjudication → RNG → state update → logging.

In [ ]:
def execute_turn(conn, spec, action: ActionIntent, seed: int | None = None) -> dict:
    """Execute a full turn: mechanical phases + GM adjudication + resolution.
    
    Returns a dict with all turn details.
    """
    # Phase 1-3: Mechanical phases (decay, multi-turn, lagged propagation)
    mech = run_mechanical_phases(conn)
    turn = mech.turn_number
    
    # Phase 4-5: GM adjudication
    state = get_all_variables(conn)
    dms = select_relevant_domain_models(spec, action)
    base_rates = compute_mechanical_base_rate(dms, action, state)
    
    messages = build_gm_messages(
        action=action,
        state=state,
        domain_models=dms,
        base_rates=base_rates,
        actor_ids=list(valid_actor_ids),
        variable_ids=list(valid_var_ids),
        mechanical_deltas=mech.all_mechanical_deltas,
    )
    
    packet, call_result = call_llm_structured(
        model=GM_MODEL,
        messages=messages,
        response_model=AdjudicationPacket,
        task='wargame_gm_adjudication',
        trace_id=TRACE_ID,
        max_budget=1.0,
    )
    
    # Normalize probabilities if needed
    prob_sum = sum(o.probability for o in packet.possible_outcomes)
    if abs(prob_sum - 1.0) > 0.001:
        packet = normalize_probabilities(packet)
    
    # Validate
    issues = validate_adjudication(packet, valid_var_ids, valid_actor_ids, base_rates)
    
    # Phase 6: RNG resolution
    outcomes_dicts = [
        {'outcome_id': o.outcome_id, 'probability': o.probability, 
         'state_transitions': [{'var_id': t.var_id, 'delta': t.delta} for t in o.state_transitions],
         'narrative': o.narrative}
        for o in packet.possible_outcomes
    ]
    chosen, rng_roll = resolve_action(conn, outcomes_dicts, turn, seed=seed)
    
    # Phase 7: Apply GM's state transitions + causal propagation
    gm_deltas = apply_action_transitions(conn, chosen['state_transitions'], turn)
    
    # Phase 8: Log
    action_id = generate_action_id()
    conn.execute(
        'INSERT INTO action_log (action_id, turn_number, actor_id, action_intent, adjudication_packet, realized_outcome_id, rng_roll) VALUES (?, ?, ?, ?, ?, ?, ?)',
        (action_id, turn, action.actor_id, action.model_dump_json(), packet.model_dump_json(), chosen['outcome_id'], rng_roll),
    )
    conn.commit()
    
    return {
        'turn': turn,
        'action': action,
        'mechanical_deltas': mech.all_mechanical_deltas,
        'outcome': chosen['outcome_id'],
        'narrative': chosen['narrative'],
        'gm_deltas': gm_deltas,
        'issues': issues,
        'cost': getattr(call_result, 'cost', 0),
    }

## Define 5 Alternating Actions

A realistic sequence: US sanctions → Iran proxies → US force posture → Iran diplomacy → US cyber op

In [ ]:
turn_actions = [
    ActionIntent(
        actor_id='actor_us',
        action_category='economic',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_sanctions'],
        intended_effect='Escalate secondary sanctions on Iranian oil exports to squeeze regime revenue.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_iran',
        action_category='kinetic',
        target_entities=['actor_us'],
        instruments_used=['inst_iran_proxies'],
        intended_effect='Escalate Hezbollah and Houthi attacks on US-allied positions to signal resolve.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_us',
        action_category='kinetic',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_force_posture'],
        intended_effect='Deploy additional carrier strike group to Persian Gulf as deterrent.',
        resource_cost=3,
    ),
    ActionIntent(
        actor_id='actor_iran',
        action_category='diplomatic',
        target_entities=['actor_us'],
        instruments_used=['inst_iran_diplomacy'],
        intended_effect='Open backchannel through Oman proposing enrichment freeze for partial sanctions relief.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_us',
        action_category='covert',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_cyber'],
        intended_effect='Deploy cyberweapon to degrade centrifuge performance at Natanz.',
        resource_cost=3,
        ambiguity_flags=['deniable_attribution'],
    ),
]

print('Planned sequence:')
for i, a in enumerate(turn_actions):
    print(f'  Turn {i+1}: [{a.actor_id}] {a.action_category} — {a.intended_effect[:60]}...')

## Execute 5 Turns

In [ ]:
results = []
total_cost = 0

for i, action in enumerate(turn_actions):
    print(f'\n{"="*70}')
    print(f'TURN {i+1}: {action.actor_id} — {action.action_category}')
    print(f'{"="*70}')
    
    result = execute_turn(conn, spec, action, seed=42+i)
    results.append(result)
    total_cost += result['cost']
    
    print(f'  Mechanical deltas: {result["mechanical_deltas"]}')
    print(f'  Outcome: {result["outcome"]}')
    print(f'  Narrative: {result["narrative"][:150]}...')
    print(f'  GM deltas: {result["gm_deltas"]}')
    if result['issues']:
        print(f'  ⚠ Issues: {result["issues"]}')
    else:
        print(f'  ✓ Valid')

print(f'\n{"="*70}')
print(f'Total cost for 5 turns: ${total_cost:.4f}')
print(f'Projected 20-turn game: ${total_cost/5*20:.2f}')

## Plot State Variable Trajectories

In [ ]:
import matplotlib.pyplot as plt

history = get_state_history(conn)

# Group variables by domain for cleaner plotting
security_vars = ['sv_nuclear_latency', 'sv_deterrence_balance', 'sv_military_tension', 
                 'sv_force_posture', 'sv_proxy_network_presence', 'sv_proxy_activity', 'sv_regional_depth']
political_vars = ['sv_sanctions_intensity', 'sv_internal_legitimacy', 'sv_elite_cohesion',
                  'sv_alliance_credibility', 'sv_diplomatic_engagement', 'sv_regime_survival_risk', 'sv_us_credibility']
perceptual_vars = ['sv_perceived_us_coercion', 'sv_perceived_iran_threat']

fig, axes = plt.subplots(3, 1, figsize=(16, 18))
fig.suptitle('State Variable Trajectories (5 turns, alternating US/Iran actions)', fontsize=14, y=0.98)

# Action annotations
action_labels = [
    (1, 'US: Sanctions'),
    (2, 'IR: Proxies'),
    (3, 'US: Force'),
    (4, 'IR: Diplomacy'),
    (5, 'US: Cyber'),
]

for ax, var_group, title in [
    (axes[0], security_vars, 'Security Variables'),
    (axes[1], political_vars, 'Political/Economic Variables'),
    (axes[2], perceptual_vars, 'Perceptual Variables'),
]:
    for var_id in var_group:
        if var_id in history:
            points = history[var_id]
            turns = [p[0] for p in points]
            values = [p[1] for p in points]
            label = var_id.replace('sv_', '')
            ax.plot(turns, values, '-o', markersize=4, label=label)
    
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Turn')
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Add action annotations
    for turn, label in action_labels:
        ax.axvline(x=turn, color='gray', linestyle='--', alpha=0.3)
        ax.text(turn, ax.get_ylim()[1], label, rotation=45, fontsize=7, ha='left', va='bottom')

plt.tight_layout()
plt.savefig('../notebooks/nb3_trajectories.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved to notebooks/nb3_trajectories.png')

## State Summary Table

In [ ]:
final = get_all_variables(conn)
initial = spec.initial_state

print(f'{"Variable":<35} {"Initial":>8} {"Final":>8} {"Change":>8}')
print('-' * 65)
for var_id in sorted(final):
    init_val = initial.get(var_id, 0)
    change = final[var_id] - init_val
    flag = ' ←' if abs(change) > 0.15 else ''
    print(f'{var_id:<35} {init_val:>8.3f} {final[var_id]:>8.3f} {change:>+8.3f}{flag}')

## Narrative Arc

In [ ]:
print('GAME NARRATIVE (5 turns)\n')
for r in results:
    print(f'--- Turn {r["turn"]}: {r["action"].actor_id} [{r["action"].action_category}] ---')
    print(f'Outcome: {r["outcome"]}')
    print(f'{r["narrative"]}\n')

## Feedback Loop Check

Does the escalation spiral emerge from the combined mechanical + GM dynamics?

In [ ]:
# Check if the expected feedback loop pattern is visible
print('Feedback loop indicators:\n')

# Sanctions should have increased (US action turn 1)
si_change = final['sv_sanctions_intensity'] - initial['sv_sanctions_intensity']
print(f'Sanctions intensity: {si_change:+.3f} {"(increased — sanctions actions)" if si_change > 0 else "(decreased — decay dominates)"}')

# Perceived coercion should have risen (causal edge from sanctions)
pc_change = final['sv_perceived_us_coercion'] - initial['sv_perceived_us_coercion']
print(f'Perceived US coercion (Iran): {pc_change:+.3f} {"(increased — sanctions → coercion edge)" if pc_change > 0 else "(decreased)"}')

# Proxy activity should have increased (Iran action turn 2)
pa_change = final['sv_proxy_activity'] - initial['sv_proxy_activity']
print(f'Proxy activity: {pa_change:+.3f} {"(increased — Iran proxy action)" if pa_change > 0 else "(decreased — decay dominates)"}')

# Perceived Iran threat should have changed (causal edge from proxy activity)
pt_change = final['sv_perceived_iran_threat'] - initial['sv_perceived_iran_threat']
print(f'Perceived Iran threat (US): {pt_change:+.3f}')

# Military tension should show the combined pressure
mt_change = final['sv_military_tension'] - initial['sv_military_tension']
print(f'Military tension: {mt_change:+.3f}')

# Nuclear latency should have decreased (momentum + possible cyber effect)
nl_change = final['sv_nuclear_latency'] - initial['sv_nuclear_latency']
print(f'Nuclear latency: {nl_change:+.3f} months')

print(f'\n--- Assessment ---')
print(f'Total LLM cost: ${total_cost:.4f}')
all_valid = all(len(r['issues']) == 0 for r in results)
print(f'All adjudications valid: {"✓" if all_valid else "✗"}')

## Summary

**Pass criteria:**
- [ ] Turn sequence executes correctly (mechanical → GM → RNG → transitions → propagation → log)
- [ ] 5 sequential turns produce coherent state trajectories (plotted)
- [ ] Feedback loop visible in the data
- [ ] No variable blows up or collapses unrealistically
- [ ] Total LLM cost < $1
- [ ] All turns logged to action_log